### Procesamiento básico de un archivo de video con YOLO v8
#### Caso 1: Conteo de personas (por frames)

In [ ]:
from ultralytics import YOLO
import cv2

def count_people_in_video(video_path):
    model = YOLO("../models/yolov8n.pt")  # or "yolov8s.pt"/"yolov8m.pt" if you have them

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {video_path}")

    frame_idx = 0
    total_people_detected = 0
    max_people_in_frame = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame, imgsz=640, conf=0.25, classes=[0], verbose=False)  # only person class
        people_this_frame = 0

        for r in results:
            if r.boxes is None:
                continue
            for box in r.boxes:
                people_this_frame += 1
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame, "person", (x1, y1 - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

        total_people_detected += people_this_frame
        max_people_in_frame = max(max_people_in_frame, people_this_frame)
        frame_idx += 1

        cv2.putText(frame, f"frame {frame_idx} people {people_this_frame}", (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

        cv2.imshow("People detection", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()
    return frame_idx, total_people_detected, max_people_in_frame

video_filename = "../data/samples/personas1.mp4"  # set your filename here (in root folder with notebook)
frame_idx, total_people_detected, max_people_in_frame = count_people_in_video(video_filename)
print("frames processed:", frame_idx)
print("total person detections (sum over frames):", total_people_detected)
print("max people in a frame:", max_people_in_frame)

frames processed: 987
total person detections (sum over frames): 5814
max people in a frame: 14


#### Caso 2: Conteo de vehículos (por frames)

In [ ]:
# Conteo de vehículos usando YOLO v8
from ultralytics import YOLO
import cv2

def count_vehicles_in_video(video_path):
    model = YOLO("../models/yolov8n.pt")  # or "yolov8s.pt"/"yolov8m.pt" if you have them

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {video_path}")

    frame_idx = 0
    total_vehicles_detected = 0
    max_vehicles_in_frame = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # COCO vehicle classes: car=2, motorcycle=3, bus=5, truck=7
        vehicle_classes = [2, 3, 5, 7]
        results = model(frame, imgsz=640, conf=0.25, classes=vehicle_classes, verbose=False)
        vehicles_this_frame = 0

        for r in results:
            if r.boxes is None:
                continue
            for box in r.boxes:
                vehicles_this_frame += 1
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)
                cv2.putText(frame, "vehicle", (x1, y1 - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

        total_vehicles_detected += vehicles_this_frame
        max_vehicles_in_frame = max(max_vehicles_in_frame, vehicles_this_frame)
        frame_idx += 1

        cv2.putText(frame, f"frame {frame_idx} vehicles {vehicles_this_frame}", (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

        cv2.imshow("Vehicle detection", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()
    return frame_idx, total_vehicles_detected, max_vehicles_in_frame

# Cambia el nombre del archivo de video según tu caso
video_filename = "../data/samples/vehiculos1.mp4"
frame_idx, total_vehicles_detected, max_vehicles_in_frame = count_vehicles_in_video(video_filename)
print("frames processed:", frame_idx)
print("total vehicle detections (sum over frames):", total_vehicles_detected)
print("max vehicles in a frame:", max_vehicles_in_frame)

frames processed: 1773
total vehicle detections (sum over frames): 12528
max vehicles in a frame: 17


#### Caso 3: Conteo de personas (frames + tracking)

In [2]:
import cv2
import numpy as np
from ultralytics import YOLO

def count_with_tracking(video_path,model_path,conf_threshold = 0.3,line_y = 300):
    model = YOLO(model_path)
    cap = cv2.VideoCapture(video_path)
    # TRACKING DATA
    count = 0
    counted_ids = set()
    track_history = {}
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        results = model.track(
            frame,
            persist=True,
            conf=conf_threshold,
            tracker="bytetrack.yaml",
            classes=[0],  # 0 = persona en COCO
            verbose=False
        )
        if results[0].boxes.id is None:
            continue
        boxes = results[0].boxes.xyxy.cpu().numpy()
        track_ids = results[0].boxes.id.cpu().numpy()
        for box, track_id in zip(boxes, track_ids):
            x1, y1, x2, y2 = map(int, box)
            cx = int((x1 + x2) / 2)
            cy = int((y1 + y2) / 2)

            # guardar historial
            if track_id not in track_history:
                track_history[track_id] = []
            track_history[track_id].append((cx, cy))
            
            # LINE CROSSING LOGIC
            if len(track_history[track_id]) >= 2:
                prev_y = track_history[track_id][-2][1]
                curr_y = cy

                # cruza la línea de arriba hacia abajo
                if prev_y < line_y and curr_y >= line_y:
                    if track_id not in counted_ids:
                        count += 1
                        counted_ids.add(track_id)

            cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
            cv2.putText(frame, f"ID {int(track_id)}", (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)

            cv2.circle(frame, (cx, cy), 4, (0,0,255), -1)

        # DRAW LINE
        cv2.line(frame, (0, line_y), (frame.shape[1], line_y), (255,0,0), 2)

        # DISPLAY COUNT
        cv2.putText(frame, f"Count: {count}", (50,50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 3)

        # SHOW
        cv2.imshow("Tracking + Counting", frame)

        if cv2.waitKey(1) & 0xFF == 27:
            break

    cap.release()
    cv2.destroyAllWindows()

video_path,model_path = "../data/samples/personas1.mp4","../models/yolov8n.pt"
count_with_tracking(video_path,model_path)